# Anomaly Comparison: LightGBM vs Imredi

## Overview

This notebook benchmarks the LightGBM-based anomaly detection pipeline against 
an external reference set provided by Imredi — the company that supplied the 
original dataset. Imredi independently applied their own detection pipeline to 
the same August 2025 test period, producing 1,194 flagged events across 494 
products over 24 days.

The comparison uses the same statistical evaluation framework defined in 
`LightGBM_Final_Methodology_Test.ipynb`, ensuring metric consistency across all 
compared systems. Two final LightGBM configurations are evaluated alongside the 
Imredi baseline: Huber Conservative z > 4.0 (recommended configuration, ~7 
alerts/day) and RMSE Strong z > 4.0 (highest recall alternative, ~30 alerts/day).

The evaluation covers four dimensions:
- **Statistical signal strength** — are flagged events statistically extreme 
  relative to each product's historical baseline?
- **Stock filter compliance** — do flagged events satisfy the domain constraint 
  that stock must be available at anomaly time and in the preceding hour?
- **Anomaly depth** — how severe are the detected sales drops relative to the 
  product norm?
- **Operational alert volume** — how many alerts per day does each system generate?

In [1]:
import pandas as pd
import numpy as np
import os

FULL_DATA_PATH = "data_v1.csv"
TEST_START = "2025-08-01"
ANOMALY_DIR = "LightGBM_fullprod_z4"
IMREDI_SAMPLE = "target_sample.csv"

ANOMALY_FILES = ["rmse_strong_z4.csv", "huber_conservative_z4.csv"]

MODEL_NAMES = [
    "Original Imredi results",
    "RMSE Strong z4",
    "Huber Conservative z4"
]

In [2]:
def evaluate_anomalies(anomaly_df, full_df, test_start="2025-08-01"):
    pre_test = full_df[full_df['date'] < test_start].copy()
    first_pos = (pre_test[pre_test['sales'] > 0]
                 .groupby('product')['date'].min()
                 .reset_index(name='first_positive_date'))
    pre_test = pre_test.merge(first_pos, on='product', how='left')
    pre_test = pre_test[pre_test['date'] >= pre_test['first_positive_date']].copy()

    stats_full = pre_test.groupby('product').agg(
        mean_sales_hist   = ('sales', 'mean'),
        std_sales_hist    = ('sales', 'std'),
        median_sales_hist = ('sales', 'median'),
    ).reset_index()

    stats_july = (pre_test[pre_test['date'].dt.month == 7]
                  .groupby('product').agg(
                      mean_sales_july = ('sales', 'mean'),
                      std_sales_july  = ('sales', 'std'),
                  ).reset_index())

    anom = anomaly_df.copy()
    anom['date']    = pd.to_datetime(anom['date'])
    anom['product'] = anom['product'].astype(str)
    anom['hour']    = anom['date'].dt.hour
    anom['weekday'] = anom['date'].dt.dayofweek

    anom = anom.merge(stats_full, on='product', how='left')
    anom = anom.merge(stats_july, on='product', how='left')

    for col in ['std_sales_hist', 'std_sales_july', 'mean_sales_hist', 'mean_sales_july']:
        anom[col] = anom[col].replace(0, np.nan).fillna(1)

    anom['z_hist'] = (anom['sales'] - anom['mean_sales_hist']) / anom['std_sales_hist']
    anom['z_july'] = (anom['sales'] - anom['mean_sales_july']) / anom['std_sales_july']
    anom['dev_pct_hist'] = (anom['sales'] - anom['mean_sales_hist']) / anom['mean_sales_hist'] * 100
    anom['dev_pct_july'] = (anom['sales'] - anom['mean_sales_july']) / anom['mean_sales_july'] * 100

    stock_prev = full_df[['date', 'product', 'stocks']].rename(columns={'stocks': 'stocks_prev', 'date': 'prev_date'})
    anom['prev_date'] = anom['date'] - pd.Timedelta(hours=1)
    anom = anom.merge(stock_prev, on=['prev_date', 'product'], how='left')

    drop_mask      = anom['sales'] < anom['mean_sales_hist']
    zero_mask      = anom['sales'] == 0
    deep_drop_mask = anom['dev_pct_hist'] < -50
    stock_ok_mask  = (anom['stocks_prev'] > 0) & (anom['stocks'] > 0)
    n = len(anom)

    return {
        'Anomalies found': n,
        'Unique products': anom['product'].nunique(),
        'Unique days': anom['date'].dt.date.nunique(),
        'Anomalies/day (avg)': round(n / max(anom['date'].dt.date.nunique(), 1), 1),

        'Mean deviation from norm (%)': round(anom.loc[drop_mask, 'dev_pct_hist'].mean(), 1),
        'Median deviation from norm (%)': round(anom.loc[drop_mask, 'dev_pct_hist'].median(), 1),
        'Mean deviation from July (%)': round(anom.loc[drop_mask, 'dev_pct_july'].mean(), 1),

        'Mean |z_hist|': round(anom['z_hist'].abs().mean(), 3),
        'Mean |z_july|': round(anom['z_july'].abs().mean(), 3),
        '% |z_hist| > 2': round((anom['z_hist'].abs() > 2).mean() * 100, 1),
        '% |z_hist| > 3': round((anom['z_hist'].abs() > 3).mean() * 100, 1),
        '% |z_july| > 2': round((anom['z_july'].abs() > 2).mean() * 100, 1),
        '% |z_july| > 3': round((anom['z_july'].abs() > 3).mean() * 100, 1),

        '% zero sales': round(zero_mask.mean() * 100, 1),
        '% drops > 50% below norm': round(deep_drop_mask.mean() * 100, 1),
        '% anomalies with stock': round(stock_ok_mask.mean() * 100, 1),

        'Zero stock at anomaly time (count)': int((anom['stocks'] == 0).sum()),
        'Zero stock previous hour (count)': int((anom['stocks_prev'] == 0).sum()),

        'Peak hour (mode)': int(anom['hour'].mode().iloc[0]) if n > 0 else None,
        '% working hours (7–22)': round(((anom['hour'] >= 7) & (anom['hour'] <= 22)).mean() * 100, 1),
        '% weekdays': round((anom['weekday'] < 5).mean() * 100, 1),
    }



In [3]:
df_full = pd.read_csv(FULL_DATA_PATH)
df_full['date'] = pd.to_datetime(df_full['date'])
df_full['product'] = df_full['product'].astype(str)

In [5]:
results = {}

anom_df = pd.read_csv(IMREDI_SAMPLE)
metrics = evaluate_anomalies(anom_df, df_full, TEST_START)
results["Original Imredi results"] = metrics

for fname, mname in zip(ANOMALY_FILES, MODEL_NAMES[1:]):
    fpath = os.path.join(ANOMALY_DIR, fname)
    anom_df = pd.read_csv(fpath)
    metrics = evaluate_anomalies(anom_df, df_full, TEST_START)
    results[mname] = metrics

comparison_df = pd.DataFrame(results)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.precision', 2)


comparison_df

,Original Imredi results,RMSE Strong z4,Huber Conservative z4
Anomalies found,1194.00,945.00,214.00
Unique products,494.00,222.00,58.00
Unique days,24.00,30.00,30.00
Anomalies/day (avg),49.80,31.50,7.10
Mean deviation from norm (%),-99.80,-94.70,-94.50
Median deviation from norm (%),-100.00,-100.00,-100.00
Mean deviation from July (%),-99.60,-96.90,-96.60
Mean |z_hist|,0.57,3.25,2.19
Mean |z_july|,0.53,2.49,1.77
% |z_hist| > 2,5.70,47.20,31.80


## Conclusions

The comparison demonstrates a clear advantage of the LightGBM-based pipeline 
over the Imredi reference across all four evaluation dimensions.

**Statistical signal strength.** The mean historical z-score of Huber Conservative 
anomalies is 2.19, and 3.25 for RMSE Strong — compared to 0.57 for Imredi. The 
proportion of anomalies exceeding |z_hist| > 2 is 31.8% (Huber) and 47.2% (RMSE 
Strong), versus 5.7% for Imredi. This means the majority of Imredi-flagged events 
fall within normal statistical variation and would not be recognisable as anomalies 
by an independent observer. The LightGBM pipeline flags events that are genuinely 
extreme relative to each product's historical baseline.

**Stock filter compliance.** All LightGBM anomalies satisfy the mandatory domain 
constraint — stock confirmed positive in both the anomaly hour and the preceding 
hour. The Imredi set contains 40 events with zero stock at anomaly time and 34 with 
zero stock in the preceding hour. These events reflect legitimate out-of-stock 
conditions rather than shelf availability failures, and should not be actioned by 
store staff. The absence of any such events in the LightGBM output is a direct 
result of the mandatory domain filter applied during detection.

**Anomaly depth.** Mean deviation from historical norm is −94.5% for Huber 
Conservative and −94.7% for RMSE Strong, versus −99.8% for Imredi. The near-100% 
deviation for Imredi is again explained by the prevalence of zero-stock events — 
when a product has no stock, sales are zero by definition, not because of a shelf 
failure. Among genuine drops (sales below norm with stock available), LightGBM 
identifies events with 41.6% (Huber) and 28.8% (RMSE Strong) classified as drops 
exceeding 50% below norm.

**Operational alert volume.** Imredi generates ~50 alerts per day. RMSE Strong 
generates ~31 alerts per day — a 37% reduction but still a substantial load for 
a discount-format store with limited staff. Huber Conservative generates ~7 alerts 
per day, an 86% reduction relative to Imredi, while maintaining 92.3% recall on 
controlled anomalies. At this volume, store staff can realistically act on every 
alert during a single shift without alert fatigue.

**Recommended configuration.** Huber Conservative at z > 4.0 is the recommended 
configuration for production deployment, as established in 
`LightGBM_Final_Methodology_Test.ipynb`. It generates the lowest operational load 
among all tested configurations, maintains full stock filter compliance, and 
produces a mean |z_hist| of 2.19 — nearly four times higher than the Imredi 
reference (0.57). RMSE Strong at z > 4.0 is the recommended alternative when 
maximising recall is the priority, for instance during promotional periods or 
seasonal peaks when missed anomalies carry a higher business cost.

In summary, the LightGBM pipeline consistently identifies anomalies with stronger 
statistical grounding, stricter domain validity, and a more operationally tractable 
alert volume than the Imredi reference. The results confirm that residual-based 
detection with learned product-level expectations substantially outperforms the 
threshold-based approach on a heterogeneous retail catalogue.